In [2]:
#!pip install pymupdf4llm

In [34]:
from pathlib import Path

In [35]:

documents_dir = Path.home() / "Documents"
file_path = documents_dir / "Спецификация из 321022-2024-00-ЭП10 v2.pdf"
print("Путь к файлу:", file_path)
print("Файл существует:", file_path.exists())

Путь к файлу: C:\Users\sv.medvedev\Documents\Спецификация из 321022-2024-00-ЭП10 v2.pdf
Файл существует: True


In [36]:
import pymupdf4llm

In [37]:
md = pymupdf4llm.to_markdown(file_path)
print(md)  # Посмотрите, как выглядят таблицы в Markdown







|Поз.|Наименование и техническая характеристика|Тип, марка.<br>обозначение документа,<br>опросного листа|Код продукции|Поставщик|Ед.<br>измерения|Кол.|Масса<br>1 ед.,<br>кг|Примечание|
|---|---|---|---|---|---|---|---|---|
||**_1. Основное электротехническое оборудование ЗПП_**||||||||
|1.1|Разъединитель 3-х полюсный 220 кВ наружной установки с 2-мя<br>заземляющими ножами, I/ном=2000 А, I/терм=40 кА<br>Комплектно с:<br>- заводской опорной металлоконструкцией (Нопоры=2900 мм);<br>- электродвигательным приводом для главных ножей;<br>- электродвигательным приводом для заземляющих ножей;<br>- с выносным блоком управления;<br>- с защитными козырьками.|по типу<br>РГНП.2-220/2000-40 УХЛ1|||компл.<br>шт<br>шт<br>шт<br>шт<br>шт|2<br>2<br>2<br>4<br>2<br>6|2500<br>1290<br>57<br>57||
|1.2|Ввод линейный U/ном=220 кВ, I/ном=2000 А, I/терм=50 кА, шт.|по типу<br>ГКЛПlll-90-252/2000 01|||шт|6|376||
|1.3|Трансформатор тока однофазный элегазовый 4-х обмоточный наружной<br>установки, U/ном=220 кВ, к

In [38]:
json_data = pymupdf4llm.to_json(file_path)

In [39]:
json_data

'{"filename": "C:\\\\Users\\\\sv.medvedev\\\\Documents\\\\Спецификация из 321022-2024-00-ЭП10 v2.pdf", "page_count": 2, "toc": [], "pages": [{"page_number": 1, "width": 1190.52001953125, "height": 841.9199829101562, "boxes": [{"x0": 47.519996643066406, "y0": 571.56005859375, "x1": 61.679996490478516, "y1": 629.4000244140625, "boxclass": "picture", "image": null, "table": null, "max_fontsize": null, "header_level": 0, "textlines": []}, {"x0": 48.119998931884766, "y0": 646.5599975585938, "x1": 63.23999786376953, "y1": 712.6799926757812, "boxclass": "picture", "image": null, "table": null, "max_fontsize": null, "header_level": 0, "textlines": []}, {"x0": 48.959999084472656, "y0": 733.3199462890625, "x1": 60.599998474121094, "y1": 804.8399658203125, "boxclass": "picture", "image": null, "table": null, "max_fontsize": null, "header_level": 0, "textlines": []}, {"x0": 99.5999984741211, "y0": 50.053653717041016, "x1": 1146.9658203125, "y1": 625.007080078125, "boxclass": "table", "image": null

In [40]:
import json
data = json.loads(json_data)
data

{'filename': 'C:\\Users\\sv.medvedev\\Documents\\Спецификация из 321022-2024-00-ЭП10 v2.pdf',
 'page_count': 2,
 'toc': [],
 'pages': [{'page_number': 1,
   'width': 1190.52001953125,
   'height': 841.9199829101562,
   'boxes': [{'x0': 47.519996643066406,
     'y0': 571.56005859375,
     'x1': 61.679996490478516,
     'y1': 629.4000244140625,
     'boxclass': 'picture',
     'image': None,
     'table': None,
     'max_fontsize': None,
     'header_level': 0,
     'textlines': []},
    {'x0': 48.119998931884766,
     'y0': 646.5599975585938,
     'x1': 63.23999786376953,
     'y1': 712.6799926757812,
     'boxclass': 'picture',
     'image': None,
     'table': None,
     'max_fontsize': None,
     'header_level': 0,
     'textlines': []},
    {'x0': 48.959999084472656,
     'y0': 733.3199462890625,
     'x1': 60.599998474121094,
     'y1': 804.8399658203125,
     'boxclass': 'picture',
     'image': None,
     'table': None,
     'max_fontsize': None,
     'header_level': 0,
     'tex

In [51]:
len(data['pages'])

2

In [62]:
spec =[]
for page in data['pages']:
    print(page.get('tables', []))
    for table in page.get('tables', []):
        print(type(table))        

[]
[]


In [65]:
from typing import List, Optional

def parse_spec_robust(pdf_path: str) -> List[List[str]]:
    """
    Извлечение спецификации с использованием JSON.
    Args:
        pdf_path: Путь к PDF файлу
    Returns:
        Список строк спецификации
    """
    cols_count = 9
    spec_data = []
    prev_spec_line = None
    first_table = True
    # 1. Сначала пробуем JSON (более структурированный)
    try:
        json_str = pymupdf4llm.to_json(pdf_path)
        json_data = json.loads(json_str)
        
        for page in json_data.get('pages', []):
            for block in page.get('boxes', []):
                if block.get('boxclass') != 'table':
                    continue
                    
                json_table_data = block.get('table', {})
                if json_table_data.get('col_count', 0) != cols_count: 
                    continue
                if json_table_data.get('row_count', 0) < 2: 
                    continue
                    
                table_data = json_table_data.get('extract', [])
                # Проверяем шапку на "Примечание"
                header = table_data[0]
                if header[8] != "Примечание":
                    continue
                if first_table: 
                    spec_data.append(header)
                    first_table = False
                  
                for row in table_data[1:]:
                    if len(row) != cols_count:
                        continue
                    if all(cell == '' for cell in row):
                        continue
                    spec_line = row
                    #normalize_row(spec_line, prev_spec_line)
                    if spec_line == [str(i) for i in range(1, cols_count+1)]: 
                        continue  # игнорируем ['1', '2', '3', '4', '5', '6', '7', '8', '9'] 
                    spec_data.append(spec_line)
                    prev_spec_line = spec_line
                
        if spec_data:
            #print(f"✓ Спецификация найдена через JSON: {len(spec_data)} строк")
            return spec_data
                    
    except Exception as e:
        print(f"⚠️ JSON метод не сработал: {e}")

In [66]:
parse_spec_robust(file_path)

[['Поз.',
  'Наименование и техническая характеристика',
  'Тип, марка. \nобозначение документа, \nопросного листа',
  'Код продукции',
  'Поставщик',
  'Ед. \nизмерения',
  'Кол.',
  'Масса \n1 ед., \n кг',
  'Примечание'],
 ['',
  '1. Основное электротехническое оборудование ЗПП',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 ['1.1',
  'Разъединитель 3-х полюсный 220 кВ наружной установки с 2-мя \nзаземляющими ножами, I/ном=2000 А, I/терм=40 кА  \nКомплектно с: \n- заводской опорной металлоконструкцией (Нопоры=2900 мм); \n- электродвигательным приводом для главных ножей; \n- электродвигательным приводом для заземляющих ножей; \n- с выносным блоком управления; \n- с защитными козырьками.',
  'по типу  \nРГНП.2-220/2000-40 УХЛ1',
  '',
  '',
  'компл. \nшт \nшт \nшт \nшт \nшт',
  '2 \n2 \n2    \n4 \n2 \n6',
  '2500 \n1290 \n57    \n57',
  ''],
 ['1.2',
  'Ввод линейный U/ном=220 кВ, I/ном=2000 А, I/терм=50 кА, шт.',
  'по типу  \nГКЛПlll-90-252/2000 01',
  '',
  '',
  'шт',
  '6',
  '376